# 57 — Vision Fine-Tuning Mini-Lab

## Goal
Fine-tune a **vision-language model** (VLM) on a narrow image classification
task using SFT with image inputs.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **VLM fine-tuning** | SFT with image+text inputs |
| **Image preprocessing** | How VLMs handle image tokens |
| **LoRA on VLMs** | Adapt vision-language models efficiently |

## Approach
Since full VLM fine-tuning requires significant VRAM, we demonstrate the
**concept** using a lightweight approach: fine-tune a text model on detailed
image descriptions (simulating what a VLM does). We also show the real VLM
training code pattern for reference.

## Stack
- `transformers` + `peft` + `trl`
- `Pillow` for image handling
- Model: **Qwen2.5-0.5B-Instruct** (text-based demo)

In [ ]:
import os, warnings
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

import torch
print(f"PyTorch {torch.__version__}  CUDA: {torch.cuda.is_available()}")

## Step 1 — Simulated Image Classification Dataset

We create text descriptions of product images (as a VLM would encode them)
and train the model to classify products.

In a real VLM setup, you'd pass actual images. Here we simulate
the vision encoder's output as text descriptions.

In [ ]:
CATEGORIES = ["electronics", "clothing", "furniture", "food", "books"]

image_descriptions = [
    # Electronics
    {"desc": "A silver laptop with a 15-inch display, showing a desktop screen. Black keyboard visible. Apple logo on the back. Placed on a wooden desk.", "category": "electronics", "details": {"type": "laptop", "brand": "Apple", "color": "silver", "condition": "new"}},
    {"desc": "A pair of white wireless earbuds in an open charging case. LED indicator glowing green. Small, compact design. AirPods-style shape.", "category": "electronics", "details": {"type": "earbuds", "brand": "generic", "color": "white", "condition": "new"}},
    {"desc": "A large curved monitor with thin bezels. Displaying a colorful gradient wallpaper. 27-inch screen. USB-C port visible on the side.", "category": "electronics", "details": {"type": "monitor", "brand": "Samsung", "color": "black", "condition": "new"}},
    {"desc": "A smartphone with a cracked screen. Gold color, large camera bump on back. Volume buttons on left side. Fingerprint smudges visible.", "category": "electronics", "details": {"type": "smartphone", "brand": "generic", "color": "gold", "condition": "damaged"}},
    {"desc": "A mechanical keyboard with RGB backlighting. Cherry MX switches. Detachable USB-C cable. Compact 65% layout.", "category": "electronics", "details": {"type": "keyboard", "brand": "generic", "color": "black", "condition": "new"}},

    # Clothing
    {"desc": "A navy blue blazer hanging on a wooden hanger. Two-button front. Notch lapels. Interior label reads 'Hugo Boss'. Slim fit cut.", "category": "clothing", "details": {"type": "blazer", "brand": "Hugo Boss", "color": "navy blue", "condition": "new"}},
    {"desc": "A pair of distressed light-wash jeans folded on a white surface. Ripped knees. Five-pocket design. Straight leg cut.", "category": "clothing", "details": {"type": "jeans", "brand": "generic", "color": "light blue", "condition": "new"}},
    {"desc": "A red winter parka with a fur-trimmed hood. Multiple zippered pockets. Down-filled. Waist drawstring visible. Size tag shows XL.", "category": "clothing", "details": {"type": "parka", "brand": "generic", "color": "red", "condition": "new"}},
    {"desc": "A pair of white canvas sneakers with rubber soles. Classic low-top design. Lace-up closure. Minimal branding. Slightly worn toe area.", "category": "clothing", "details": {"type": "sneakers", "brand": "generic", "color": "white", "condition": "used"}},

    # Furniture
    {"desc": "A mid-century modern armchair with walnut legs. Teal upholstered seat and back. Angled design. Placed in a living room with a rug.", "category": "furniture", "details": {"type": "armchair", "brand": "generic", "color": "teal", "condition": "new"}},
    {"desc": "A standing desk with a bamboo top and black steel frame. Electric height adjustment buttons visible. Cable management tray underneath.", "category": "furniture", "details": {"type": "desk", "brand": "generic", "color": "natural/black", "condition": "new"}},
    {"desc": "A white 5-shelf bookcase against a gray wall. Filled with books and small plants. Particle board construction. Some sagging on middle shelf.", "category": "furniture", "details": {"type": "bookcase", "brand": "IKEA-style", "color": "white", "condition": "used"}},

    # Food
    {"desc": "A jar of organic peanut butter with a yellow label. Glass jar, metal lid. 'No sugar added' text visible. 16oz size. Brand: Whole Earth.", "category": "food", "details": {"type": "peanut butter", "brand": "Whole Earth", "color": "brown/yellow", "condition": "sealed"}},
    {"desc": "A box of green tea bags. 20 individually wrapped sachets. Japanese text on packaging. Cherry blossom design on box. Matcha powder visible.", "category": "food", "details": {"type": "tea", "brand": "Japanese import", "color": "green", "condition": "sealed"}},
    {"desc": "A bundle of fresh asparagus held together with a rubber band. Green stalks, purple-tinged tips. Sitting on a wooden cutting board.", "category": "food", "details": {"type": "vegetable", "brand": "n/a", "color": "green", "condition": "fresh"}},

    # Books
    {"desc": "A hardcover book with a dark blue cover. Title: 'Deep Learning' by Ian Goodfellow. Yellow geometric pattern on cover. 800+ pages, thick spine.", "category": "books", "details": {"type": "textbook", "brand": "MIT Press", "color": "blue", "condition": "new"}},
    {"desc": "A worn paperback novel. Stephen King's 'The Shining'. Creased spine, dog-eared pages. Movie tie-in cover with Jack Nicholson.", "category": "books", "details": {"type": "novel", "brand": "Anchor Books", "color": "black/red", "condition": "used"}},
    {"desc": "A children's picture book with bright illustrations. 'Goodnight Moon' by Margaret Wise Brown. Board book format. Chewed corner.", "category": "books", "details": {"type": "children's book", "brand": "Harper", "color": "green/multi", "condition": "used"}},
]

import json

train_data = image_descriptions[:15]
eval_data = image_descriptions[15:]
print(f"Train: {len(train_data)}, Eval: {len(eval_data)}")
print(f"Categories: {CATEGORIES}")

In [ ]:
from datasets import Dataset

SYS_MSG = (
    "You are a product image classifier. Given a description of a product image, "
    f"classify it into one of: {', '.join(CATEGORIES)}. "
    "Also extract: type, color, condition (new/used/damaged). "
    "Output as JSON: {\"category\": ..., \"type\": ..., \"color\": ..., \"condition\": ...}"
)

formatted = []
for ex in train_data:
    output = json.dumps({"category": ex["category"], "type": ex["details"]["type"],
                         "color": ex["details"]["color"], "condition": ex["details"]["condition"]})
    formatted.append({"messages": [
        {"role": "system", "content": SYS_MSG},
        {"role": "user", "content": f"Classify this product image:\n{ex['desc']}"},
        {"role": "assistant", "content": output},
    ]})

train_ds = Dataset.from_list(formatted)
print(f"Dataset: {train_ds}")

## Step 2 — Fine-Tune

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = "./vision_classifier_model"

trainer = SFTTrainer(
    model=MODEL_ID,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        max_steps=60,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        warmup_steps=5,
        max_length=384,
        bf16=True,
        logging_steps=10,
        gradient_checkpointing=True,
        report_to="none",
    ),
    train_dataset=train_ds,
    peft_config=LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
        task_type="CAUSAL_LM",
    ),
)

print("Fine-tuning vision classifier...")
trainer.train()
trainer.save_model(OUTPUT_DIR)
print("✅ Done!")

## Step 3 — Evaluate

In [ ]:
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer

model = AutoPeftModelForCausalLM.from_pretrained(OUTPUT_DIR, torch_dtype=torch.bfloat16, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

correct = 0
for ex in eval_data:
    messages = [
        {"role": "system", "content": SYS_MSG},
        {"role": "user", "content": f"Classify this product image:\n{ex['desc']}"},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=100, temperature=0.1, do_sample=True)
    response = tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

    try:
        pred = json.loads(response)
        pred_cat = pred.get("category", "")
    except json.JSONDecodeError:
        pred_cat = response.strip()

    match = pred_cat == ex["category"]
    correct += int(match)
    print(f"{'✅' if match else '❌'} Expected: {ex['category']:12s} Got: {pred_cat:12s} | {ex['desc'][:60]}...")

print(f"\nAccuracy: {correct}/{len(eval_data)} ({correct/len(eval_data)*100:.0f}%)")

## Reference: Real VLM Fine-Tuning Code

For actual vision fine-tuning with image inputs, here's the pattern
using a VLM like Qwen2.5-VL:

```python
# Real VLM fine-tuning (requires ~16GB+ VRAM)
from trl import SFTConfig, SFTTrainer
from datasets import load_dataset

# Dataset with 'image' column (PIL images) + 'messages' column
dataset = load_dataset("your-image-dataset", split="train")

trainer = SFTTrainer(
    model="Qwen/Qwen2.5-VL-3B-Instruct",  # VLM model
    args=SFTConfig(
        max_length=None,  # Important: don't truncate image tokens!
        output_dir="./vlm_finetuned",
        bf16=True,
    ),
    train_dataset=dataset,
    peft_config=LoraConfig(r=16, ...),
)
trainer.train()
```

Key differences from text SFT:
- `max_length=None` to avoid truncating image tokens
- Dataset must have `image` or `images` column
- VLM models are larger (3B+), need more VRAM

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **VLM fine-tuning** | Same SFTTrainer, different model + image data |
| **max_length=None** | Critical for VLMs — don't truncate image tokens |
| **Text proxy** | Use image descriptions when VLM is too large |
| **Multi-label output** | Category + type + color + condition in one JSON |

## 🔧 Production Extensions
- Use a real VLM (Qwen2.5-VL, LLaVA) with image inputs
- Add more categories and brand recognition
- Use image augmentation for robustness
- Benchmark against CLIP/ViT classifiers